# Data Preprocessing & Feature Engineering

This notebook runs the full preprocessing pipeline and generates features for model training:
- Implicit rating construction from behavioral events
- User-level feature aggregation
- Item-level popularity features
- Session-based temporal features
- Train/Val/Test split (temporal)

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import RetailRocketDataLoader
from src.feature_engine import FeatureEngine

%matplotlib inline

## 1. Run Data Pipeline

In [ ]:
loader = RetailRocketDataLoader()
loader.run_pipeline()

print(f"Interactions: {len(loader.interactions):,}")
print(f"Users: {len(loader.user_to_idx):,}")
print(f"Items: {len(loader.item_to_idx):,}")
loader.interactions.head()

## 2. Feature Engineering

In [ ]:
item_features = loader.build_item_features()
engine = FeatureEngine(loader.events, loader.interactions, item_features)

user_features, item_pop_features, enriched = engine.run_pipeline()

print(f"User features shape: {user_features.shape}")
print(f"Item features shape: {item_pop_features.shape}")
print(f"Enriched interactions shape: {enriched.shape}")

In [ ]:
print("=== User Features ===")
user_features.describe().round(2)

In [ ]:
print("=== Item Features ===")
item_pop_features.describe().round(2)

## 3. Session Features

In [ ]:
session_features = engine.compute_temporal_features()
print(f"Session features shape: {session_features.shape}")
session_features.describe().round(2)

## 4. Interaction Matrix

In [ ]:
interaction_matrix = loader.build_interaction_matrix()
print(f"Shape: {interaction_matrix.shape}")
print(f"Non-zero entries: {interaction_matrix.nnz:,}")
print(f"Sparsity: {1 - interaction_matrix.nnz / (interaction_matrix.shape[0] * interaction_matrix.shape[1]):.4%}")

## 5. Save Processed Data

In [ ]:
loader.save_processed()

import os
PROJECT_ROOT = os.path.abspath('..')
user_features.to_parquet(os.path.join(PROJECT_ROOT, 'data/processed/user_features.parquet'), index=False)
item_pop_features.to_parquet(os.path.join(PROJECT_ROOT, 'data/processed/item_pop_features.parquet'), index=False)

print("All processed data saved to data/processed/")

import os
processed_dir = os.path.join(PROJECT_ROOT, 'data/processed')
for f in os.listdir(processed_dir):
    size = os.path.getsize(os.path.join(processed_dir, f)) / 1024
    print(f"  {f}: {size:.1f} KB")